# EDA: Azure VM Trace (vmtable) — d2ql data preprocessing

Notebook de exploración del trace de Azure usado como carga de trabajo (workload) para el
entorno de scheduling DDQN de **d2ql**.

**Fuente:** `data/workload.csv.gz` → symlink a `trace_data_vmtable_vmtable.csv.gz`
(Azure Public Dataset V1, tabla de VMs; Cortez et al. 2017).

**Hallazgo clave:** el archivo **NO tiene header**. La primera fila son datos hasheados (IDs),
no nombres de columna. Se carga con `header=None`.

## Mapeo de columnas (validado contra la documentación oficial + inspección de los datos)

| idx | nombre | tipo | descripción |
|---|---|---|---|
| 0 | subscription_id | hash | ID de suscripción (encriptado) |
| 1 | deployment_id | hash | ID de deployment (encriptado) |
| 2 | vm_id | hash | ID de VM (encriptado) |
| 3 | vm_created_ts | int | timestamp creación VM (s, desde t=0) |
| 4 | vm_deleted_ts | int | timestamp borrado VM (s) |
| 5 | max_cpu_pct | float | máximo CPU % (0-100) |
| 6 | avg_cpu_pct | float | promedio CPU % |
| 7 | p95_max_cpu_pct | float | p95 del máximo CPU % |
| 8 | vm_category | cat | Delay-insensitive / Interactive / Unknown |
| 9 | core_count | cat | 2, 4, 8, 24, >24 |
| 10 | memory_gb | cat | 2, 4, 8, 32, 64, >64 |

Filas totales (pasada completa previa): **2,695,548**. Tamaño: 418 MB comprimido.

In [3]:
import pandas as pd
import numpy as np
import os

import pathlib
ROOT = pathlib.Path(os.getcwd())
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = str(ROOT / "data" / "workload.csv.gz")
RAW_COLUMNS = [
    "subscription_id", "deployment_id", "vm_id",
    "vm_created_ts", "vm_deleted_ts",
    "max_cpu_pct", "avg_cpu_pct", "p95_max_cpu_pct",
    "vm_category", "core_count", "memory_gb",
]
print("pandas", pd.__version__)
print("file exists:", os.path.exists(DATA))
assert os.path.exists(DATA), "Falta data/workload.csv.gz"


pandas 3.0.5
file exists: True


## 1) Carga rápida de una muestra
El archivo completo son 418 MB / ~2.7M filas. Para iterar cargamos una muestra;
para el análisis completo se puede quitar `nrows` o leer por chunks (`chunksize=`).

In [7]:
SAMPLE_ROWS = 500_000
df = pd.read_csv(DATA, compression="gzip", header=None,
                 names=RAW_COLUMNS, nrows=SAMPLE_ROWS)
print("shape:", df.shape)
df.head()


shape: (500000, 11)


,subscription_id,deployment_id,vm_id,vm_created_ts,vm_deleted_ts,max_cpu_pct,avg_cpu_pct,p95_max_cpu_pct,vm_category,core_count,memory_gb
0,71fJw0x+SDRdAxKPwLyHZhTgQpYw2afS6tjJhfT6kHnmLH...,GB6uQC1NSArW5n+TtOybL7GQ1yByjuWtZnsj+5QccZ525R...,2sh/ZjaYdfpslv4iYBfNzFe4rs982kHVvNGJGeQ8MIBCDr...,558300,1673700,91.776885,0.728879,20.759630,Delay-insensitive,8,32
1,rKggHO/04j31UFy65mDTwtjdMQL/G03xWfl3xGeiilB4/W...,ub4ty8ygwOECrIz7eaZ/9hDwnCsERvZ3nJJ03sDSpD85et...,+ZraIDUNaWYDZMBiBtZm7xSjr+j3zcHGjup1+wyKxHFmyJ...,424500,425400,37.879261,3.325358,37.879261,Unknown,4,32
2,YrR8gPtBmfNaOdnNEW5If1SdTqQgGQHEnLHGPjySt53bKW...,9LrdYRcUfGbmL2fFfLR/JUg2OTkjGRe3iluwIhDRPnPDPa...,GEyIElfPSFupze8T+T1niQMepeqG88VpLNuxUMyIDbz8VF...,1133100,1133700,0.304368,0.220553,0.304368,Unknown,4,32
3,xzQ++JF1UAkh70CDhmzkiOo+DQn+E2TLErCFKEmSswv1pl...,0XnZZ8sMN5HY+Yg+0dykYB5oenlgsrCpzpgFSvn/MX42Ze...,7aCQS6fPUw9rwCPiqvghk/WCEbMV3KgNJjA+sssdfY5Ybl...,0,2591400,98.573424,30.340054,98.212503,Interactive,2,4
4,vZEivnhabRmImDr+JqKqZnpIM3WxtypwoxjfjnklR/idyR...,HUGaZ+piPP4eHjycCBki2yq0raJywdzrVuriR6nQceH3hA...,/s/D5VtTQDxyS6wq7N/VQAMczx61Ny1Ut3a3iFmDSOCXxp...,228300,229800,82.581449,13.876299,82.581449,Unknown,2,4


## 2) Tipos de datos y primeras filas

In [9]:
df.dtypes

subscription_id        str
deployment_id          str
vm_id                  str
vm_created_ts        int64
vm_deleted_ts        int64
max_cpu_pct        float64
avg_cpu_pct        float64
p95_max_cpu_pct    float64
vm_category            str
core_count             str
memory_gb              str
dtype: object

## 3) Valores nulos por columna

In [10]:
nulos = df.isna().sum()
print(nulos.to_string())
print("\nTotal celdas nulas:", int(nulos.sum()))


subscription_id    0
deployment_id      0
vm_id              0
vm_created_ts      0
vm_deleted_ts      0
max_cpu_pct        0
avg_cpu_pct        0
p95_max_cpu_pct    0
vm_category        0
core_count         0
memory_gb          0

Total celdas nulas: 0


## 4) Distribución de columnas categóricas
`vm_category`, `core_count`, `memory_gb`.

In [12]:
for col in ["vm_category", "core_count", "memory_gb"]:
    print("==", col, "==")
    print(df[col].value_counts(dropna=False).to_string())
    print()


== vm_category ==
vm_category
Unknown              456125
Delay-insensitive     29497
Interactive           14378

== core_count ==
core_count
2      294199
4      152883
8       36079
24      14927
>24      1912

== memory_gb ==
memory_gb
8      188530
32     157237
4       76345
2       60375
64      15601
>64      1912



## 5) Estadísticos de columnas numéricas
Timestamps de creación/borrado y métricas de CPU.

In [13]:
num_cols = ["vm_created_ts", "vm_deleted_ts",
            "max_cpu_pct", "avg_cpu_pct", "p95_max_cpu_pct"]
pd.set_option("display.width", 120)
df[num_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
vm_created_ts,500000.0,1.183007e+06,797382.480113,0.0,472200.000000,1.152600e+06,1.859400e+06,2.591400e+06
vm_deleted_ts,500000.0,1.398485e+06,798897.565097,0.0,701400.000000,1.395000e+06,2.138100e+06,2.591400e+06
max_cpu_pct,500000.0,5.562494e+01,36.962632,0.0,17.822146,6.165973e+01,9.368233e+01,1.000000e+02
avg_cpu_pct,500000.0,1.557837e+01,18.735315,0.0,2.037584,8.209806e+00,2.218183e+01,9.937626e+01
p95_max_cpu_pct,500000.0,4.714745e+01,35.768954,0.0,10.005564,4.847753e+01,8.241769e+01,1.000000e+02


## 6) Duración de las VMs y validación de integridad
`duration_s = vm_deleted_ts - vm_created_ts`. Verificamos que creación <= borrado.

In [14]:
df["duration_s"] = df["vm_deleted_ts"] - df["vm_created_ts"]
neg = int((df["duration_s"] < 0).sum())
zero = int((df["duration_s"] == 0).sum())
print(f"filas muestra      : {len(df):,}")
print(f"duración < 0       : {neg}  (creado despues de borrado -> anomalia)")
print(f"duración == 0      : {zero}")
print("\nduration_s describe:")
print(df["duration_s"].describe())


filas muestra      : 500,000
duración < 0       : 0  (creado despues de borrado -> anomalia)
duración == 0      : 26518

duration_s describe:
count    5.000000e+05
mean     2.154780e+05
std      6.677047e+05
min      0.000000e+00
25%      6.000000e+02
50%      1.800000e+03
75%      1.200000e+04
max      2.591400e+06
Name: duration_s, dtype: float64


## 7) Correlación entre métricas numéricas

In [15]:
corr_cols = num_cols + ["duration_s"]
df[corr_cols].corr(numeric_only=True)


,vm_created_ts,vm_deleted_ts,max_cpu_pct,avg_cpu_pct,p95_max_cpu_pct,duration_s
vm_created_ts,1.000000,0.650072,-0.101065,0.061843,0.040250,-0.416414
vm_deleted_ts,0.650072,1.000000,0.132102,-0.023359,-0.030223,0.420158
max_cpu_pct,-0.101065,0.132102,1.000000,0.610305,0.848584,0.278751
avg_cpu_pct,0.061843,-0.023359,0.610305,1.000000,0.734680,-0.101803
p95_max_cpu_pct,0.040250,-0.030223,0.848584,0.734680,1.000000,-0.084228
duration_s,-0.416414,0.420158,0.278751,-0.101803,-0.084228,1.000000


## 8) Relevancia para el entorno DDQN de d2ql

El scheduler aprende una política de colocación/asignación de VMs. Las columnas útiles como
**estado (state)** o **características del workload** son:

- `core_count`, `memory_gb` → recursos solicitados (la acción típica es a qué host/nodo asignar).
- `vm_category` → prioridad / clase de servicio (Interactive suele ser latency-sensitive).
- `max_cpu_pct` / `avg_cpu_pct` / `p95_max_cpu_pct` → utilización real, útil para el reward
  (¿qué tan bien empaquetamos los nodos?).
- `duration_s` (derivada) → duración de la VM; relevante para consolidación y desfragmentación.
- `vm_created_ts` / `vm_deleted_ts` → ventana de vida; útil para generar arrival/departure
  events en la simulación CloudSimPlus.

Los hashes (`subscription_id`, `deployment_id`, `vm_id`) normalmente NO son features, pero
`vm_id` sirve para agrupar/ordenar eventos. `subscription_id`/`deployment_id` pueden usarse
para estudiar la hipótesis H3 (generalización cross-workload) agrupando por deployment.

**Siguiente paso:** convertir esto en `scripts/preprocess.py` que produzca el dataset limpio
(listo para el entorno Gymnasium) — derivar `duration_s`, codificar `vm_category`,
hacer one-hot de `core_count`/`memory_gb`, y filtrar anomalías de duración.